# Single-Qubit Simulation

A qubit is described by a state vector $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ with $|\alpha|^2 + |\beta|^2 = 1$. QuSim stores it as a length-2 complex array and represents each gate as a 2x2 unitary matrix, applied with `gate @ state`. Measuring the qubit picks an outcome with probability given by the squared amplitude and collapses the state to that basis state.

| Concept | Module |
| --- | --- |
| Gates I, X, Y, Z, H, S, T and rotations Rx, Ry, Rz, Phase | `qusim.gates` |
| State construction, normalization and probabilities | `qusim.state` |
| Sampling and state collapse | `qusim.measure` |
| Histograms and Bloch sphere plots | `qusim.viz` |

The examples assume that `qusim` is importable, either installed or run from the repository root.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qusim import __version__, gates, measure, state, viz

print("qusim version:", __version__)
np.set_printoptions(precision=4, suppress=True)


## The starting state |0>

The computational basis state |0> has amplitude 1 in the first entry and 0 in the second. Its probabilities are therefore `[1, 0]`, and its Bloch vector points to the north pole.


In [ ]:
psi = state.zero_state(1)
print("state:        ", psi)
print("probabilities:", state.probabilities(psi))
print("bloch:        ", viz.bloch_coordinates(psi))


## Single-qubit gates

Every gate is available both as a matrix (`gates.h @ psi`) and as a helper that applies it (`gates.H(psi)`).


In [ ]:
for name, matrix in [("X", gates.x), ("H", gates.h), ("Z", gates.z)]:
    out = matrix @ psi
    print(f"{name}|0> = {out}   probabilities = {state.probabilities(out)}")


In [ ]:
psi_x = gates.X(psi)        # |1>
psi_plus = gates.H(psi)     # |+>

for label, s in [("X|0> = |1>", psi_x), ("H|0> = |+>", psi_plus)]:
    print(f"{label:12s} state={s}  probs={state.probabilities(s)}  bloch={viz.bloch_coordinates(s)}")


## Unitarity and norm preservation

A valid quantum gate satisfies $U^{\dagger}U = I$. Applying a unitary gate preserves the norm of the state, so the total probability always remains 1.


In [ ]:
names = ["I", "X", "Y", "Z", "H", "S", "T"]
matrices = [gates.i, gates.x, gates.y, gates.z, gates.h, gates.s, gates.t]

for name, matrix in zip(names, matrices):
    out = matrix @ psi_plus
    unitary = np.allclose(matrix.conj().T @ matrix, np.eye(2))
    print(f"{name}: unitary={unitary}  norm={np.linalg.norm(out):.6f}")


## Rotations and phase

`rx(theta)`, `ry(theta)`, `rz(theta)` and `phase(phi)` return 2x2 matrices, with angles measured in radians.


In [ ]:
theta = np.pi / 2
for name, matrix in [("Rx", gates.rx(theta)), ("Ry", gates.ry(theta)), ("Rz", gates.rz(theta))]:
    out = matrix @ psi
    print(f"{name}(pi/2)|0> = {out}  bloch={viz.bloch_coordinates(out)}")


In [ ]:
phi = np.pi / 2
out = gates.phase(phi) @ psi_plus
print("phase(pi/2)|+> =", out, " bloch=", viz.bloch_coordinates(out))


Rotating |+> about Z by 90 degrees produces |+i>, which points along the +Y axis of the Bloch sphere.

## Visualizing states on the Bloch sphere

The Bloch vector $(x, y, z)$ of a pure state is computed from its amplitudes, and the state is drawn as an arrow inside the unit sphere.


In [ ]:
states = [
    ("|0>", psi),
    ("|1>", psi_x),
    ("|+>", psi_plus),
    ("|+i>", gates.phase(np.pi / 2) @ psi_plus),
]

fig = plt.figure(figsize=(12, 10))
for i, (label, s) in enumerate(states, start=1):
    ax = fig.add_subplot(2, 2, i, projection="3d")
    viz.plot_bloch(s, ax=ax)
    ax.set_title(f"{label}  bloch={viz.bloch_coordinates(s)}")
plt.tight_layout()
plt.show()


## Sampling measurement outcomes

`sample` performs `shots` measurements and returns a dictionary of bitstring counts. Seeding the random generator makes the result reproducible.


In [ ]:
rng = np.random.default_rng(42)
counts = measure.sample(psi_plus, shots=1000, rng=rng)
print(counts)

viz.plot_histogram(counts)
plt.show()


The |+> state has equal probabilities for both outcomes, so the counts land close to 500/500.

## Single-shot measurement and collapse

`measure` returns one outcome bitstring together with the state vector after collapse.


In [ ]:
outcome, collapsed = measure.measure(psi_plus, rng=np.random.default_rng(7))
print("outcome:        ", outcome)
print("collapsed state:", collapsed)
print("bloch:          ", viz.bloch_coordinates(collapsed))


After collapse the state lies on the Z axis, because a measurement always leaves the qubit in a computational basis state.

## Normalizing a state

`normalize` rescales any non-zero vector so that its probabilities sum to 1.


In [ ]:
raw = np.array([1, 1j])
normalized = state.normalize(raw)
print("normalized:   ", normalized)
print("probabilities:", state.probabilities(normalized))
print("bloch:        ", viz.bloch_coordinates(normalized))
